# Parser loading and input diagnostic (development only)
Run this first: four controlled arms compare original/profile input with NF4/FP16 loading.
Uses fresh development tasks, the original 1.5B adapter, fixed base revision and identical decoding/packages.
No Stage 2 outputs, GitHub push, 3B adapter, generator or repair model required.
Select GPU T4 x2, Internet on, attach the original 1.5B adapter, then Save & Run All in a fresh session.
Reference code is used only after each parser response for scoring; it never enters model messages.
This is a diagnosis, not a promise of improved accuracy. Download the final ZIP for review before running all six systems.

In [ ]:
import base64, hashlib, json, os, subprocess, sys, zlib
from pathlib import Path
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
PAYLOAD_SHA='98cca4a215c0cc352acc8c99b1789e02061393ceacc96154aff61a20358ee332'
PAYLOAD=(
'...')
raw=zlib.decompress(base64.b64decode(PAYLOAD))
assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA
ROOT=WORK/('stage2_independent_v2_source_'+PAYLOAD_SHA[:12])
for name,encoded_source in json.loads(raw).items():
    source=base64.b64decode(encoded_source)
    relative=Path(name)
    assert not relative.is_absolute() and '..' not in relative.parts
    path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():assert path.read_bytes()==source, f'Changed embedded source: {path}'
    else:path.write_bytes(source)
PACKAGES=WORK/'stage2_independent_v2_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')
print('Source and raw inputs:',PAYLOAD_SHA)


In [ ]:
PRESET='proposed'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
# PEFT probes Kaggle's optional torchao package even for this ordinary LoRA adapter.
# Check adapter injection on a tiny CPU model before downloading the large models.
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
ADAPTER=None
if PRESET in ['proposed','no_contract_to_generator']:
    spec=json.loads((ROOT/'eval/stage2_independent_v2/adapter.json').read_text())
    matches=[]
    for config in Path('/kaggle/input').rglob('adapter_config.json'):
        try:
            if json.loads(config.read_text()).get('base_model_name_or_path')==spec['model']:matches.append(config.parent)
        except ValueError:pass
    assert len(matches)==1, f'Attach exactly one original 1.5B adapter; found {matches}'
    ADAPTER=matches[0]
    for name,expected in spec['sha256'].items():assert hashlib.sha256((ADAPTER/name).read_bytes()).hexdigest()==expected, f'Wrong adapter: {name}'
print('System:',PRESET,'Adapter:',ADAPTER)


In [ ]:
OUT=WORK/'stage2_parser_diagnostic'
assert not OUT.exists(), 'Start a fresh Kaggle session; do not mix arms from different runs.'
OUT.mkdir()
ARMS=['original_nf4','profile_nf4','original_fp16','profile_fp16']
for arm in ARMS:
    command=[sys.executable,'-m','eval.stage2_independent_v2.diagnostic','--arm',arm,'--adapter',str(ADAPTER),'--out',str(OUT/arm),'--packages',str(PACKAGES)]
    with (OUT/(arm+'.log')).open('w') as log:
        process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');log.write(line);log.flush()
        code=process.wait()
    assert code==0, f'{arm} failed; download its log and partial ZIP for diagnosis.'
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.diagnostic','--summarize','--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
from IPython.display import FileLink,Markdown,display
display(Markdown((OUT/'SUMMARY.md').read_text()))
display(FileLink(str(OUT.with_suffix('.zip'))))
print('COMPLETE: four controlled development arms')
